# 1 - Import

In [16]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import QuantLib as ql
import yfinance as yf
import sdmx
from pandas_datareader import data as web
from datetime import datetime
import vectorbt as vbt
from datetime import timedelta

# 2 - Fetch Data

## API Request Def

In [15]:
def get_eurusd_price(start_date, end_date):
  Data = pd.DataFrame()
  eurusd = yf.download("EURUSD=X", start=start_date, end=end_date)
  print(eurusd.head())
  Data["Open"] = eurusd["Open"]
  Data["High"] = eurusd["High"]
  Data["Low"] = eurusd["Low"]
  Data["Close"] = eurusd["Close"]
  Data["Volume"] = eurusd["Volume"]
  Data.index = eurusd.index
  return Data

get_eurusd_price("01-01-2025","01-10-2025")

C:\Users\Valérian Darmenté\AppData\Local\Temp\ipykernel_18208\1307799927.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

Failed to get ticker 'EURUSD=X' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: C:\Users\Valérian Darmenté\PyCharmMiscProject\Trading_Project\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['EURUSD=X']: YFTzMissingError('possibly delisted; no timezone found')


Empty DataFrame
Columns: [(Adj Close, EURUSD=X), (Close, EURUSD=X), (High, EURUSD=X), (Low, EURUSD=X), (Open, EURUSD=X), (Volume, EURUSD=X)]
Index: []


,Open,High,Low,Close,Volume
Date,,,,,


# 3 - Pricing Options

## European Simple Option

In [7]:
def european_fx_option_pricing(option_type, S, K, r_usd, r_eur, T, sigma, today):
    """
    Price a European FX option (e.g., EUR/USD) using Black-Scholes-Merton model.

    Parameters:
    - option_type: 'call' or 'put' (call on EUR/USD = right to buy EUR, sell USD)
    - S: Spot exchange rate (e.g., EUR/USD spot rate in USD per EUR)
    - K: Strike exchange rate
    - r_usd: USD interest rate (domestic rate for EUR/USD)
    - r_eur: EUR interest rate (foreign rate for EUR/USD)
    - T: Time to maturity in years
    - sigma: Volatility of the exchange rate
    - today: QuantLib Date object for evaluation date

    Returns:
    - Tuple of (option_price, delta, gamma, theta, vega)
    """
    # Input validation
    if S <= 0 or K <= 0:
        raise ValueError("Spot price (S) and strike price (K) must be positive.")
    if sigma <= 0:
        raise ValueError("Volatility (sigma) must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity (T) must be positive.")

    # 1. Set the evaluation date
    ql.Settings.instance().evaluationDate = today

    # 2. Define the option type
    if option_type.lower() == 'call':
        option_type_ql = ql.Option.Call  # Call on EUR/USD: right to buy EUR
    elif option_type.lower() == 'put':
        option_type_ql = ql.Option.Put  # Put on EUR/USD: right to sell EUR
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    # 3. Create the option object
    payoff = ql.PlainVanillaPayoff(option_type_ql, K)
    # Calculate maturity date (use calendar for accuracy)
    exercise_date = today + ql.Period(int(T * 360), ql.Days)  # Approximate T in days
    exercise = ql.EuropeanExercise(exercise_date)
    european_option = ql.VanillaOption(payoff, exercise)

    # 4. Define the market data
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    # Domestic rate (USD for EUR/USD)
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_usd, ql.Actual365Fixed()))
    # Foreign rate (EUR for EUR/USD)
    dividend_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_eur, ql.Actual365Fixed()))

    vol_handle = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), sigma, ql.Actual365Fixed()))

    # 5. Create the Black-Scholes process for FX
    bsm_process = ql.BlackScholesMertonProcess(spot_handle, dividend_handle, rate_handle, vol_handle)

    # 6. Create the pricing engine
    engine = ql.AnalyticEuropeanEngine(bsm_process)

    # 7. Set the pricing engine for the option
    european_option.setPricingEngine(engine)

    # 8. Calculate the option price and Greeks
    option_price = european_option.NPV()  # Price in USD per EUR
    delta = european_option.delta()  # Delta w.r.t. spot exchange rate
    gamma = european_option.gamma()  # Gamma w.r.t. spot exchange rate
    theta = european_option.theta() / 365  # Annualized theta (USD per EUR per year)
    vega = european_option.vega() / 100   # Vega per 1% change in volatility

    return option_price, delta, gamma, theta, vega


In [8]:
T = 1/12

In [10]:
def pricing_atmf_straddle(Data):
  # Initialize DataFrame to store straddle price and Greeks
  Option_Price_List = pd.DataFrame(
      columns=[f"Straddle_{int(T*12)}M_ATMF_USD", "Delta", "Gamma", "Theta", "Vega"],
      index=Data.index
  )

  vol = Data["EURUSD"].pct_change().dropna().std() * np.sqrt(252)  # Annualized volatility
  print(f"Calculated volatility: {vol}")

  # Iterate over DataFrame
  for index, row in Data.iterrows():
      S = row["EURUSD"]
      K = row["EURUSD"]*np.exp((row["USD_Rate_1Y"]-row["EUR_Rate_1Y"]+1/2*vol**2)*T)
      r = row["USD_Rate_1Y"]
      q = row["EUR_Rate_1Y"]
      sigma = vol
      today = ql.Date(index.day, index.month, index.year)

      try:
          # Price call and put options
          option_price_call_atm, delta_call, gamma_call, theta_call, vega_call = european_fx_option_pricing(
              "call", S, K, r, q, T, sigma, today
          )
          option_price_put_atm, delta_put, gamma_put, theta_put, vega_put = european_fx_option_pricing(
              "put", S, K, r, q, T, sigma, today
          )
          # Calculate straddle price and Greeks
          straddle_price = option_price_call_atm + option_price_put_atm
          straddle_delta = delta_call + delta_put
          straddle_gamma = gamma_call + gamma_put
          straddle_theta = theta_call + theta_put
          straddle_vega = vega_call + vega_put

          # Store in DataFrame
          Option_Price_List.loc[index] = [
              straddle_price,
              straddle_delta,
              straddle_gamma,
              straddle_theta,
              straddle_vega
          ]
      except Exception as e:
          print(f"Error pricing for date {index}: {e}")
          Option_Price_List.loc[index] = [None, None, None, None, None]

  for i in Option_Price_List.columns:
      Data[i] = Option_Price_List[i]
  return Data


Data = get_eurusd_price("01-01-2025","01-10-2025")
print(Data.head())
print(Data.columns)
Data = pricing_atmf_straddle(Data).copy()
print(Data.head())
print(Data.columns)

# Créer le graphique avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=Data.index, y=Data[f"Straddle_{int(T*12)}M_ATMF_USD"], mode='lines', name='Upper Bound'))

fig.update_layout(
    title="Option Price Over Time",
    xaxis_title="Date",
    yaxis_title="Option Price"
)

fig.show()

C:\Users\Valérian Darmenté\AppData\Local\Temp\ipykernel_18208\3585667790.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

Failed to get ticker 'EURUSD=X' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: C:\Users\Valérian Darmenté\PyCharmMiscProject\Trading_Project\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['EURUSD=X']: SSLError('Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: C:\\Users\\Valérian Darmenté\\PyCharmMiscProject\\Trading_Project\\.venv\\Lib\\site-packages\\certifi\\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')


KeyError: 'EURUSD'

In [ ]:
def price_evolution_for_one_straddle(underlying_asset, r, q, T,K, sigma):
  # Initialize DataFrame to store straddle price and Greeks
  Option_Price_List = pd.DataFrame(
      columns=[f"Straddle_Price_USD", "Delta", "Gamma", "Theta", "Vega"],
      index=Data.index
  )

  vol = Data["EURUSD"].pct_change().dropna().std() * np.sqrt(252)  # Annualized volatility
  print(f"Calculated volatility: {vol}")

  # Iterate over DataFrame
  for index, row in Data.iterrows():
      S = row["EURUSD"]
      K = row["EURUSD"]*np.exp((row["USD_Rate_1Y"]-row["EUR_Rate_1Y"]+1/2*vol**2)*T)
      r = row["USD_Rate_1Y"]
      q = row["EUR_Rate_1Y"]
      sigma = vol
      today = ql.Date(index.day, index.month, index.year)

      try:
          # Price call and put options
          option_price_call_atm, delta_call, gamma_call, theta_call, vega_call = european_fx_option_pricing(
              "call", S, K, r, q, T, sigma, today
          )
          option_price_put_atm, delta_put, gamma_put, theta_put, vega_put = european_fx_option_pricing(
              "put", S, K, r, q, T, sigma, today
          )
          # Calculate straddle price and Greeks
          straddle_price = option_price_call_atm + option_price_put_atm
          straddle_delta = delta_call + delta_put
          straddle_gamma = gamma_call + gamma_put
          straddle_theta = theta_call + theta_put
          straddle_vega = vega_call + vega_put

          # Store in DataFrame
          Option_Price_List.loc[index] = [
              straddle_price,
              straddle_delta,
              straddle_gamma,
              straddle_theta,
              straddle_vega
          ]
      except Exception as e:
          print(f"Error pricing for date {index}: {e}")
          Option_Price_List.loc[index] = [None, None, None, None, None]

  for i in Option_Price_List.columns:
      Data[i] = Option_Price_List[i]
  return Data

Data = pricing_atmf_straddle(Data).copy()
print(Data.head())
print(Data.columns)

# Créer le graphique avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=Data.index, y=Data[f"Straddle_{int(T*12)}M_ATMF_USD"], mode='lines', name='Upper Bound'))

fig.update_layout(
    title="Option Price Over Time",
    xaxis_title="Date",
    yaxis_title="Option Price"
)

fig.show()

Calculated volatility: 0.07544634080618155
              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2018-01-02  1.201158  1.208094  1.200855    -0.007122       0.0183   
2018-01-03  1.206345  1.206709  1.200495    -0.006969       0.0181   
2018-01-04  1.201043  1.209190  1.200495    -0.006784       0.0182   
2018-01-05  1.206884  1.208459  1.202154    -0.006753       0.0180   
2018-01-08  1.203746  1.205400  1.195972    -0.006752       0.0179   

           Straddle_1M_ATMF_USD     Delta      Gamma     Theta      Vega  
Date                                                                      
2018-01-02             0.020746 -0.001191  30.728497 -0.000345  0.002749  
2018-01-03             0.020835 -0.001176  30.595975 -0.000346  0.002761  
2018-01-04             0.020743 -0.001173  30.730596 -0.000345  0.002749  
2018-01-05             0.020844 -0.001163  30.581778 -0.000346  0.002762  
2018-01-08      

## Def Pivot Points

In [ ]:
def Pivot_Points(Data, pivot_left, pivot_right):
        try:
            from collections import deque
            def clean_deque(i, k, deq, df, key, isHigh):
                if deq and deq[0] == i - k:
                    deq.popleft()
                if isHigh:
                    while deq and df.iloc[i][key] > df.iloc[deq[-1]][key]:
                        deq.pop()
                else:
                    while deq and df.iloc[i][key] < df.iloc[deq[-1]][key]:
                        deq.pop()

            data = Data[["High", "Low"]].copy()
            data['H'] = False
            data['L'] = False

            win_size = pivot_left + pivot_right + 1
            deqHigh = deque()
            deqLow = deque()

            max_idx = 0
            min_idx = 0
            i = 0
            j = pivot_left
            pivot_low = None
            pivot_high = None

            for index, row in data.iterrows():
                if i < win_size:
                    clean_deque(i, win_size, deqHigh, data, 'High', True)
                    clean_deque(i, win_size, deqLow, data, 'Low', False)
                    deqHigh.append(i)
                    deqLow.append(i)

                    if data.iloc[i]['High'] > data.iloc[max_idx]['High']:
                        max_idx = i
                    if data.iloc[i]['Low'] < data.iloc[min_idx]['Low']:
                        min_idx = i

                    if i == win_size - 1:
                        if data.iloc[max_idx]['High'] == data.iloc[j]['High']:
                            data.at[data.index[j], 'H'] = True
                            pivot_high = data.iloc[j]['High']
                        if data.iloc[min_idx]['Low'] == data.iloc[j]['Low']:
                            data.at[data.index[j], 'L'] = True
                            pivot_low = data.iloc[j]['Low']
                else:
                    j += 1
                    clean_deque(i, win_size, deqHigh, data, 'High', True)
                    clean_deque(i, win_size, deqLow, data, 'Low', False)
                    deqHigh.append(i)
                    deqLow.append(i)

                    if data.iloc[deqHigh[0]]['High'] == data.iloc[j]['High']:
                        data.at[data.index[j], 'H'] = True
                        pivot_high = data.iloc[j]['High']
                    if data.iloc[deqLow[0]]['Low'] == data.iloc[j]['Low']:
                        data.at[data.index[j], 'L'] = True
                        pivot_low = data.iloc[j]['Low']

                data.at[data.index[j], 'Last_High_Value'] = pivot_high
                data.at[data.index[j], 'Last_Low_Value'] = pivot_low
                i += 1

            # Low pivot calculation
            lows_list = []
            broken_lows = []
            first_value_low = True
            data["Low_Pivot"] = np.nan

            for idx, row in data.iterrows():
                lows_list = [x for x in lows_list if not np.isnan(x)]
                last_low = row['Last_Low_Value']
                low = row['Low']

                if pd.notna(last_low):
                    if first_value_low:
                        lows_list.append(last_low)
                        data.at[idx, "Low_Pivot"] = last_low
                        first_value_low = False
                        continue

                    if not lows_list:
                        if last_low not in broken_lows:
                            lows_list.append(last_low)
                            data.at[idx, "Low_Pivot"] = last_low
                    elif len(lows_list) > 1:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                        if last_low not in broken_lows:
                            if last_low != lows_list[-1]:
                                lows_list.append(last_low)
                            data.at[idx, "Low_Pivot"] = lows_list[-1]
                        else:
                            data.at[idx, "Low_Pivot"] = lows_list[-1]
                    else:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                            if last_low not in broken_lows:
                                lows_list.append(last_low)
                                data.at[idx, "Low_Pivot"] = last_low
                            else:
                                data.at[idx, "Low_Pivot"] = None
                        else:
                            if last_low not in broken_lows:
                                if last_low != lows_list[-1]:
                                    lows_list.append(last_low)
                                data.at[idx, "Low_Pivot"] = lows_list[-1]
                            else:
                                data.at[idx, "Low_Pivot"] = lows_list[-1]
                elif not first_value_low:
                    if lows_list:
                        if low < lows_list[-1]:
                            broken_lows.append(lows_list.pop())
                        data.at[idx, "Low_Pivot"] = lows_list[-1] if lows_list else None

                    # -------------------------------
                    # High pivot calculation
                    highs_list = []
                    broken_highs = []
                    first_value_high = True
                    data["High_Pivot"] = np.nan

                    for idx, row in data.iterrows():
                        highs_list = [x for x in highs_list if not np.isnan(x)]
                        last_high = row['Last_High_Value']
                        high = row['High']

                        if pd.notna(last_high):
                            if first_value_high:
                                highs_list.append(last_high)
                                data.at[idx, "High_Pivot"] = last_high
                                first_value_high = False
                                continue

                            if not highs_list:
                                if last_high not in broken_highs:
                                    highs_list.append(last_high)
                                    data.at[idx, "High_Pivot"] = last_high
                            elif len(highs_list) > 1:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                if last_high not in broken_highs:
                                    if last_high != highs_list[-1]:
                                        highs_list.append(last_high)
                                    data.at[idx, "High_Pivot"] = highs_list[-1]
                                else:
                                    data.at[idx, "High_Pivot"] = highs_list[-1]
                            else:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                    if last_high not in broken_highs:
                                        highs_list.append(last_high)
                                        data.at[idx, "High_Pivot"] = last_high
                                    else:
                                        data.at[idx, "High_Pivot"] = None
                                else:
                                    if last_high not in broken_highs:
                                        if last_high != highs_list[-1]:
                                            highs_list.append(last_high)
                                        data.at[idx, "High_Pivot"] = highs_list[-1]
                                    else:
                                        data.at[idx, "High_Pivot"] = highs_list[-1]
                        elif not first_value_high:
                            if highs_list:
                                if high > highs_list[-1]:
                                    broken_highs.append(highs_list.pop())
                                data.at[idx, "High_Pivot"] = highs_list[-1] if highs_list else None

            colonnes_a_supprimer = ['Last_High_Value', 'Last_Low_Value']
            data = data.drop(colonnes_a_supprimer, axis=1)

            Data["Low_Pivot"] = data["Low_Pivot"]
            Data["High_Pivot"] = data["High_Pivot"]

            return Data

        except Exception as e:
            raise

## Get Pivot Point Data

In [ ]:
left = 10
right = 10
Data = Pivot_Points(Data, right, left)
print(Data.tail())
print(Data.columns)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=Data.index, y=Data['High'], mode='lines', name='EUR Rate 3M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['Low'], mode='lines', name='EUR Rate 3M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['High_Pivot'], mode='lines', name='EUR Rate 6M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['Low_Pivot'], mode='lines', name='EUR Rate 1Y'))
fig1.update_layout(
    title=f'Pivot Point EUR ({left},{right})',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Maturity',
    template='plotly_white'
)
fig1.show()

              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2025-06-06  1.145383  1.145869  1.137230     0.018325       0.0414   
2025-06-09  1.140784  1.144034  1.138758     0.018281       0.0413   
2025-06-10  1.142805  1.144833  1.137462     0.018161       0.0412   
2025-06-11  1.143746  1.149095  1.140602     0.017834       0.0408   
2025-06-12  1.150933  1.162872  1.150285     0.017840       0.0406   

           Straddle_1M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2025-06-06             0.019741  -0.00109  32.157498 -0.000327  0.002616   
2025-06-09             0.019662 -0.001088  32.287265 -0.000325  0.002606   
2025-06-10             0.019697 -0.001088  32.230493 -0.000326   0.00261   
2025-06-11             0.019714 -0.001085  32.204835 -0.000326  0.002612   
2025-06-12             0.019837 -0.001077  32.003707 

# 4 - Get Signal

In [ ]:
def signals(Data):

    high_pivot_changed = Data["High_Pivot"] > Data["High_Pivot"].shift(1)
    low_pivot_changed = Data["Low_Pivot"] < Data["Low_Pivot"].shift(1)

    signals = []
    for i in range(len(Data)):
        if high_pivot_changed[i]:
            signals.append(1)
        elif low_pivot_changed[i]:
            signals.append(-1)
        else:
            signals.append(0)

    Data["Signals"] = signals
    return Data

Data = signals(Data)
value_counts = Data["Signals"].value_counts(dropna=False)
print(value_counts)

print(Data.tail())

Signals
 0    1739
 1      52
-1      41
Name: count, dtype: int64
              EURUSD      High       Low  EUR_Rate_1Y  USD_Rate_1Y  \
Date                                                                 
2025-06-06  1.145383  1.145869  1.137230     0.018325       0.0414   
2025-06-09  1.140784  1.144034  1.138758     0.018281       0.0413   
2025-06-10  1.142805  1.144833  1.137462     0.018161       0.0412   
2025-06-11  1.143746  1.149095  1.140602     0.017834       0.0408   
2025-06-12  1.150933  1.162872  1.150285     0.017840       0.0406   

           Straddle_1M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2025-06-06             0.019741  -0.00109  32.157498 -0.000327  0.002616   
2025-06-09             0.019662 -0.001088  32.287265 -0.000325  0.002606   
2025-06-10             0.019697 -0.001088  32.230493 -0.000326   0.00261   
2025-06-11             0.019714 -0.001085  32.204835 -0.000326

/tmp/ipython-input-15-3888941173.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-15-3888941173.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



## Def Show Backesting Results

# 5 - Strategy Pipeline

## Def Get Orders Futures

In [ ]:
def Get_Orders_Futures(Data):
    #['Open', 'High', 'Low', 'Close', 'Volume', 'Low_Pivot', 'High_Pivot', 'Signals', 'Leverage', 'TP_pct', 'SL_pct', 'Size']


    # Ensure all inputs are Pandas Series
    index = pd.Series(Data.index)
    price = pd.Series(Data["Close"])
    signals = pd.Series(Data["Signals"])
    Leverage = pd.Series(Data["Leverage"])
    SL_pct = pd.Series(Data["SL_pct"])
    TP_pct = pd.Series(Data["TP_pct"])
    Size = pd.Series(Data["Size"])

    order_id = 1
    orders = []
    i = 0
    trade_win = 0
    trade_nbr = 0

    while i < len(index):
        #---------------------------------------------------------------------------
        if signals.iloc[i] == 1:  # LONG SIGNAL
            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': Size[i],
                'price': price.iloc[i],
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': price.iloc[i]}
            })
            order_id += 1
            trade_nbr += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            SL_price = price.iloc[i]*(1-SL_pct[i])
            TP_price = price.iloc[i]*(1+TP_pct[i])
            for j in range(i + 1, len(index)):
                # First level hit triggers exit
                if price.iloc[j] <= SL_price:
                    orders.append({
                        'timestamp': index.iloc[j],
                        'size': -Size[i],
                        'price': price.iloc[i] + (SL_price - price.iloc[i])*Leverage[i],
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': price.iloc[i]}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif price.iloc[j] >= TP_price:
                    orders.append({
                        'timestamp': index.iloc[j],
                        'size': -Size[i],
                        'price': price.iloc[i] + (TP_price - price.iloc[i])*Leverage[i],
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': price.iloc[i]}
                    })
                    order_id += 1
                    trade_win += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': -Size[i],
                    'price': price.iloc[i] + (price.iloc[-1] - price.iloc[i])*Leverage[i],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': price.iloc[i]}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        #---------------------------------------------------------------------------
        elif signals.iloc[i] == -1:  # SHORT SIGNAL
            # Entry order
            orders.append({
                'timestamp': index.iloc[i],
                'size': Size[i],
                'price': price.iloc[i],
                'order_id': order_id,
                'custom': {'type': "entry", 'entry_price': price.iloc[i]}
            })
            order_id += 1
            trade_nbr += 1
            #-----------------------------------------------------------------------
            # Search for exit in the future
            exit_found = False
            SL_price = price.iloc[i]*(1+SL_pct[i])
            TP_price = price.iloc[i]*(1-TP_pct[i])
            for j in range(i + 1, len(index)):
                if price.iloc[j] >= SL_price:
                    orders.append({
                        'timestamp': index.iloc[j],
                        'size': -Size[i],
                        'price': price.iloc[i] + (SL_price - price.iloc[i])*Leverage[i],
                        'order_id': order_id,
                        'custom': {'type': "exit_SL", 'entry_price': price.iloc[i]}
                    })
                    order_id += 1
                    exit_found = True
                    break
                elif price.iloc[j] <= TP_price:
                    orders.append({
                        'timestamp': index.iloc[j],
                        'size': -Size[i],
                        'price': price.iloc[i] + (TP_price - price.iloc[i])*Leverage[i],
                        'order_id': order_id,
                        'custom': {'type': "exit_TP", 'entry_price': price.iloc[i]}
                    })
                    order_id += 1
                    trade_win += 1
                    exit_found = True
                    break
            if not exit_found:
                # If neither SL nor TP is hit, exit at last available price
                orders.append({
                    'timestamp': index.iloc[-1],
                    'size': -Size[i],
                    'price': price.iloc[i] + (price.iloc[-1] - price.iloc[i])*Leverage[i],
                    'order_id': order_id,
                    'custom': {'type': "exit_EOD", 'entry_price': price.iloc[i]}
                })
                order_id += 1
            #-----------------------------------------------------------------------
        i += 1
        #---------------------------------------------------------------------------
    # Convert to DataFrame
    orders_df = pd.DataFrame(orders)
    orders_df['timestamp'] = pd.to_datetime(orders_df['timestamp'])
    orders_df = orders_df.groupby('timestamp').agg({'size': 'sum', 'price': 'first'}).sort_index()
    # You need to define "close" or replace it with the relevant series for reindexing
    # Here, I'll assume you want to use the same index for reindexing as provided at input
    sizes = orders_df['size'].reindex(index, fill_value=0)
    prices = orders_df['price'].reindex(index)
    win_rate = trade_win / trade_nbr
    return orders_df, sizes, prices, win_rate

## Show Backtest Results

In [ ]:
def show_backtesting_result(portfolio):
    # Get asset names (always a list)
    assets = list(portfolio.wrapper.columns)
    general_stats = portfolio.stats()

    # Handle per-asset stats
    asset_stats = []
    if len(assets) == 1:
        # Single asset: use portfolio directly
        asset_stats.append(portfolio.stats())
    else:
        # Multi-assets: use portfolio[asset]
        for asset in assets:
            asset_stats.append(portfolio[asset].stats())

    # Build all unique stat names
    all_stat_names = sorted(set(general_stats.index) | set.union(*(set(a.index) for a in asset_stats)))

    # Build data dict
    data = {
        'General': [general_stats.get(stat, "") for stat in all_stat_names]
    }
    for i, asset in enumerate(assets):
        data[asset] = [asset_stats[i].get(stat, "") for stat in all_stat_names]

    df_stats = pd.DataFrame(data, index=all_stat_names)
    df_stats.index.name = 'Stats'
    df_stats_reset = df_stats.reset_index()

    # Tabulate display
    try:
        from tabulate import tabulate
        print(tabulate(df_stats_reset, headers='keys', tablefmt='fancy_grid', showindex=False, floatfmt=".6f"))
    except ImportError:
        print("Install tabulate with: pip install tabulate")
        print(df_stats_reset.to_string(index=False))

    # Plots for each asset
    for asset in assets:
        # For single asset, don't specify column
        if len(assets) == 1:
            portfolio.drawdowns.plot().show()
            portfolio.plot_underwater().show()
            portfolio.positions.plot().show()
        else:
            portfolio.drawdowns.plot(column=asset).show()
            portfolio.plot_underwater(column=asset).show()
            portfolio.positions.plot(column=asset).show()
    return df_stats_reset

## Plot Return RR/Win Rate Grid

In [18]:
def plot_rr_win_grid(RR_list , Win_Rate_list):
    RRmin = (int(min(RR_list)*100)-1*100)/100 if (int(min(RR_list)*100)-1*100)/100 > 0 else 0
    RRmax = (int(max(RR_list)*100)+1*100)/100 if (int(max(RR_list)*100)+1*100)/100 < 10 else 10
    WWmin = (int(min(Win_Rate_list)*1000)-1*100)/1000 if (int(min(Win_Rate_list)*1000)-1*100)/1000 > 0 else 0
    WWmax = (int(max(Win_Rate_list)*1000)+1*100)/1000 if (int(max(Win_Rate_list)*1000)+1*100)/1000 < 1 else 1

    RR_edges = np.arange(RRmin, RRmax, 0.01)
    Win_edges = np.arange(WWmin, WWmax, 0.001)

    RR_centers = (RR_edges[:-1] + RR_edges[1:]) / 2
    Win_centers = (Win_edges[:-1] + Win_edges[1:]) / 2

    RR_grid, Win_grid = np.meshgrid(RR_centers, Win_centers, indexing='ij')
    expected = RR_grid * Win_grid - (1 - Win_grid)

    # Discrete colors and boundaries
    bins = [-0.5, -0.2, -0.05, 0.05, 0.2, 0.5, np.max(expected)+1] # Add max+1 to include max value
    colors = [
    'rgba(255,99,132,0.5)',   # light red
    'rgba(255,179,71,0.5)',   # light orange
    'rgba(255,255,153,0.5)',  # light yellow
    'rgba(144,238,144,0.5)',  # light green
    'rgba(135,206,250,0.5)'   # light blue
]
    labels = ['< -50%', '-20% ', '0%', '20%', '> 50%']

    expected_digitized = np.digitize(expected, bins, right=False) - 1 # values: 0 to 4

    expected_points = [rr * wr - (1 - wr) for rr, wr in zip(RR_list, Win_Rate_list)]
    expected_labels = [f"{(ev*100):.2f}%" for ev in expected_points]

    # Build a discrete colorscale for Plotly
    # Positions must be evenly spaced for the discrete classes (0, 0.25, 0.5, 0.75, 1.0)
    colorscale = [
        [0.00, colors[0]],
        [0.25, colors[1]],
        [0.50, colors[2]],
        [0.75, colors[3]],
        [1.00, colors[4]],
    ]
    tickvals = [i for i in range(5)]
    ticktext = labels

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        x=RR_centers,
        y=Win_centers * 100,
        z=expected_digitized.T,
        colorscale=colorscale,
        zmin=0, zmax=4,
        colorbar=dict(
            title="Return",
            tickvals=tickvals,
            ticktext=ticktext
        ),
        customdata=expected.T*100,
        hovertemplate="Return: %{customdata:.2f}%<br>RR: %{x:.2f}<br>Win Rate: %{y:.2f}%<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=RR_list,
        y=[100 * wr for wr in Win_Rate_list],
        mode='markers+text',
        marker=dict(color='black', size=10),
        text=expected_labels,
        showlegend=False,
        textposition="top center",
        hovertemplate="Return: %{text}<br>RR: %{x:.2f}<br>Win Rate: %{y:.2f}%<extra></extra>"
    ))

    # Add y = 2/(x+1) curve (x: RR, y: Win Rate %)
    RR_curve = np.linspace(RRmin, RRmax-0.01, 300)
    Win_curve = 1 / (RR_curve+1)
    # Filter to stay in [0,1]
    mask = (Win_curve >= 0) & (Win_curve <= 1)

    fig.add_trace(go.Scatter(
        x=RR_curve[mask],
        y=Win_curve[mask]*100,
        mode='lines',
        line=dict(color='black', width=2, dash='dash'),
        showlegend=False,
        hovertemplate="Return: 0%<br>RR: %{x:.2f}<br>Win Rate: %{y:.2f}%<extra></extra>"
    ))

    fig.update_layout(
        xaxis=dict(title='Risk-Reward Ratio (RR)', range=[RRmin,RRmax-0.01], dtick=1),
        yaxis=dict(title='Win Rate (%)', range=[WWmin*100, WWmax*100-0.1], dtick=10),
        title='Return from RR / Win Rate Grid'
    )
    fig.show()

plot_rr_win_grid([3,2],[0.3,0.5])

## Variables

In [ ]:
def get_Data(start_date = "2018-01-01", end_date = "2025-06-13", pivot_left = 10, pivot_right = 10, SL_pct = 0.005, TP_pct = 0.02, leverage = 10, Size = 1):
  Data_EURUSD = pd.DataFrame()
  Data_EURUSD = get_eurusd_price(start_date, end_date)
  #--------------------------------------------------
  Data = Pivot_Points(Data_EURUSD, right, left)
  #--------------------------------------------------
  Data_EURUSD = signals(Data_EURUSD)
  #--------------------------------------------------
  Data_EURUSD["SL_pct"] = SL_pct
  Data_EURUSD["TP_pct"] = TP_pct
  #--------------------------------------------------
  Data_EURUSD["Leverage"] = leverage
  Data_EURUSD["Size"] = Data_EURUSD["Signals"] * Size
  #--------------------------------------------------
  return Data_EURUSD

In [ ]:
Data_EURUSD = get_Data(start_date = "2018-01-01", end_date = "2025-06-13", pivot_left = 10, pivot_right = 10, SL_pct = 0.005, TP_pct = 0.02, leverage = 10, Size = 1)
print("Final data Shape:", Data_EURUSD.shape)
print("Final data Columns:", Data_EURUSD.columns)
print("Final data Tail:\n", Data_EURUSD.tail())

/tmp/ipython-input-116-3770260002.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


Final Data Shape: (1940, 12)
Final Data Columns: Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Low_Pivot', 'High_Pivot',
       'Signals', 'SL_pct', 'TP_pct', 'Leverage', 'Size'],
      dtype='object')
Final Data Tail:
                 Open      High       Low     Close  Volume  Low_Pivot  \
Date                                                                    
2025-06-06  1.145383  1.145869  1.137230  1.145383       0   1.107506   
2025-06-09  1.140784  1.144034  1.138758  1.140784       0   1.107506   
2025-06-10  1.142805  1.144833  1.137462  1.142805       0   1.107506   
2025-06-11  1.143746  1.149095  1.140602  1.143746       0   1.107506   
2025-06-12  1.150933  1.162872  1.150285  1.150933       0   1.107506   

            High_Pivot  Signals  SL_pct  TP_pct  Leverage  Size  
Date                                                             
2025-06-06    1.154654        0   0.005    0.02        10     0  
2025-06-09    1.154654        0   0.005    0.02        10     0  


/tmp/ipython-input-15-3888941173.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-15-3888941173.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



## Backtest

In [ ]:
def backtest_stategy(Data, init_cash = 100, fees = 0.001, slippage = 0.0001):
    #-------------------------------------------------------------------------------
    orders_df1, sizes1, prices1, win_rate1 = Get_Orders_Futures(Data)
    #orders_df2, sizes2, prices2, win_rate2 = Get_Orders_Futures(data)

    #print(orders_df1.tail())
    #print(orders_df2.tail())

    sizes = pd.DataFrame({
        "EURUSD_1": sizes1.astype(float),
        "EURUSD_2": sizes2.astype(float)
    })
    prices = pd.DataFrame({
        "EURUSD_1": prices1.astype(float),
        "EURUSD_2": prices2.astype(float)
    })
    close = pd.DataFrame({
        "EURUSD_1": Data["Close"]#,"EURUSD_2": data["Close"]
    })
    #-------------------------------------------------------------------------------
    # Build the portfolio
    portfolio =  vbt.Portfolio.from_orders(
        close=close,
        size=sizes1,
        price=prices1,
        size_type='amount',
        init_cash=100,
        fees=0.001,
        slippage=0.0001
    )
    #-------------------------------------------------------------------------------
    # Build the portfolio
    portfolio =  vbt.Portfolio.from_orders(
        close=close,
        size=sizes,
        price=prices,
        size_type='amount',
        init_cash=100,
        fees=0.001,
        slippage=0.0001
    )
    return portfolio, win_rate1

In [ ]:
portfolio, win_rate = backtest_stategy(Data, init_cash = 100, fees = 0.001, slippage = 0.0001)
#-------------------------------------------------------------------------------
backtesting_stats = show_backtesting_result(portfolio)

/tmp/ipython-input-150-1506278904.py:26: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:36: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:37: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:43: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated 

╒════════════════════════════╤═════════════════════╤═════════════════════╤═════════════════════╕
│ Stats                      │ General             │ EURUSD_1            │ EURUSD_2            │
╞════════════════════════════╪═════════════════════╪═════════════════════╪═════════════════════╡
│ Avg Losing Trade Duration  │ 12.6                │ 12.6                │ 12.6                │
├────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│ Avg Losing Trade [%]       │ -4.823956376429702  │ -4.823956376429702  │ -4.823956376429702  │
├────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│ Avg Winning Trade Duration │ 16.06896551724138   │ 16.06896551724138   │ 16.06896551724138   │
├────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│ Avg Winning Trade [%]      │ 19.46288257988411   │ 19.46288257988411   │ 19.46288257988411   │
├────────────────────────────┼

## Parameters Optimization

In [ ]:
# Define parameter ranges
pivot_left_values = [10,20]
pivot_right_values = [10,20]
SL_pct_values = np.arange(0.01, 0.04, 0.01)
TP_pct_values = np.arange(0.01, 0.06, 0.01)
leverage_values = range(1, 11, 2)
Size_values = [1]

print(len(pivot_left_values)*len(pivot_right_values)*len(SL_pct_values)*len(TP_pct_values)*len(leverage_values)*len(Size_values))

300


In [ ]:
import numpy as np

# Define parameter ranges
pivot_left_values = [20]
pivot_right_values = [20]
SL_pct_values = np.arange(0.01, 0.04, 0.01)
TP_pct_values = np.arange(0.01, 0.06, 0.01)
leverage_values = range(1, 11, 2)
Size_values = [1]

results = []

for pivot_left in pivot_left_values:
    for pivot_right in pivot_right_values:
        for SL_pct in SL_pct_values:
            for TP_pct in TP_pct_values:
                for leverage in leverage_values:
                    for Size in Size_values:
                        Data = get_Data(
                            start_date="2018-01-01",
                            end_date="2025-06-13",
                            pivot_left=pivot_left,
                            pivot_right=pivot_right,
                            SL_pct=SL_pct,
                            TP_pct=TP_pct,
                            leverage=leverage,
                            Size=Size
                        )
                        portfolio, win_rate = backtest_stategy(
                            Data,
                            init_cash=100,
                            fees=0.001,
                            slippage=0.0001
                        )
                        results.append({
                            'pivot_left': pivot_left,
                            'pivot_right': pivot_right,
                            'SL_pct': SL_pct,
                            'TP_pct': TP_pct,
                            'leverage': leverage,
                            'Size': Size,
                            'final_portfolio': portfolio[-1],
                            'win_rate': win_rate
                        })

# Sort results by best final portfolio value
best_results = sorted(results, key=lambda x: x['final_portfolio'], reverse=True)

# Print top 5 combinations
for i, res in enumerate(best_results[:5]):
    print(
        f"Rank {i+1}: "
        f"pivot_left={res['pivot_left']}, "
        f"pivot_right={res['pivot_right']}, "
        f"SL_pct={res['SL_pct']}, "
        f"TP_pct={res['TP_pct']}, "
        f"leverage={res['leverage']}, "
        f"Size={res['Size']}, "
        f"final_portfolio={res['final_portfolio']:.2f}, "
        f"win_rate={res['win_rate']:.2%}"
    )


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:43: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:44: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-150-1506278904.py:79: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys w

TypeError: '<' not supported between instances of 'Portfolio' and 'Portfolio'